[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/44.%20The%20Carbon%20Footprint%20Modeling%20Problem/P44-Tier-1_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 44. The Carbon Footprint Modeling Problem

## Tier 1: The Pen-and-Paper Method — Mathematical Formulation

### Goal

Build and solve a mixed-integer optimization model that assigns supplier-to-plant and plant-to-customer shipments to transportation modes. The model minimizes a weighted combination of transportation and facility emissions and transportation cost, subject to shipment, vehicle-capacity, facility-activation, and carbon-budget constraints.

### Inputs, outputs, and scope

**Inputs** are the network nodes and their coordinates, supplier availability, customer demand, facility emissions, transportation capacities and per-kilometre factors, objective weights, and the total carbon budget. **Outputs** are the selected shipment links and modes, shipment quantities, active plants, emissions and cost totals, and the solver status. A carbon-budget sensitivity analysis repeats the same model with different budget limits.

This is a capacitated supply-chain network-flow model, not a vehicle-sequencing model: it does not decide the order in which a vehicle visits multiple nodes. A selected link and mode represents one vehicle movement on that link, with shipment quantity limited by that vehicle's capacity. The model uses a single planning period, Euclidean distance, deterministic inputs, and fixed daily emissions for each active plant. It does not include production costs, vehicle fleet availability, multiple trips on the same link and mode, time windows, or emissions that vary with shipment weight.

### Modeling assumptions

- Shipments may travel only from suppliers to plants and from plants to customers; there are no direct supplier-to-customer links or other transshipment links.
- Supplier quantities are available supply limits; all available supply need not be shipped. Every customer's demand must be met.
- A selected link-mode movement incurs emissions and cost proportional to its distance, independent of the quantity carried. Its shipment quantity cannot exceed that mode's capacity.
- A plant incurs its fixed daily emissions when active. The model activates a plant only when it handles shipments, because activating an unused plant cannot improve the objective.
- The coordinates define distances in kilometres. The example's coordinates are the source of the distances used by the model; the separate illustrative distances sometimes used in descriptions are not additional model inputs.
- The objective weights combine emissions in kilograms of CO₂-equivalent and monetary cost. They are decision-maker weights, not unit conversions.

### Solution sequence

1. Define the network entities and transportation modes.
2. Calculate the Euclidean distance between each pair of nodes.
3. Create binary link-mode decisions, continuous shipment quantities, continuous plant-throughput variables, and binary plant-activation decisions.
4. Minimize the weighted emissions-and-cost objective while satisfying shipment and carbon constraints.
5. Report the solution and inspect how the carbon budget affects the same optimization model.

The mixed-integer linear program is solved with the CBC solver through PuLP. A reported optimal status means the solver found an optimal solution for this encoded model; it does not establish that the model includes every feature of a real supply chain. Mixed-integer optimization can require substantial computation as the number of candidate links, modes, and binary decisions grows.

## 1. Mathematical model

A shipment decision has two parts: whether a vehicle movement is selected on a link, and how much product that movement carries. The binary decision lets the model charge the fixed emissions and cost of using a movement; the continuous quantity enforces supply, demand, and capacity. Plant activation connects shipment flow to fixed facility emissions.

Let $S$, $P$, $C$, and $V$ denote the sets of suppliers, plants, customers, and vehicle modes. Let $A$ contain only supplier-to-plant and plant-to-customer links. For each $(i,j)\in A$ and $v\in V$, define $x_{ijv}\in\{0,1\}$ as 1 when mode $v$ is used on link $(i,j)$, and let $q_{ijv}\geq 0$ be the quantity shipped on that movement. For each plant $p\in P$, let $y_p\geq 0$ be its throughput and $z_p\in\{0,1\}$ indicate whether it is active.

The objective balances total emissions against transportation cost:

$$\min\quad \alpha\left[\sum_{(i,j)\in A}\sum_{v\in V} d_{ij}e_vx_{ijv}+\sum_{p\in P} f_pz_p\right]+\beta\sum_{(i,j)\in A}\sum_{v\in V}d_{ij}c_vx_{ijv}.$$

Here, $d_{ij}$ is the distance in kilometres, $e_v$ is the mode's emissions factor in kg CO₂e/km, $c_v$ is its cost factor in currency/km, $f_p$ is the plant's fixed daily emissions in kg CO₂e, and $\alpha$ and $\beta$ are the objective weights. The carbon budget applies to emissions, not to the weighted objective:

$$\sum_{(i,j)\in A}\sum_{v\in V}d_{ij}e_vx_{ijv}+\sum_{p\in P}f_pz_p\leq B.$$

The shipment and capacity constraints are:

$$\sum_{p\in P}\sum_{v\in V}q_{spv}\leq a_s \qquad \forall s\in S,$$

$$\sum_{p\in P}\sum_{v\in V}q_{pcv}=r_c \qquad \forall c\in C,$$

$$\sum_{s\in S}\sum_{v\in V}q_{spv}=y_p=\sum_{c\in C}\sum_{v\in V}q_{pcv} \qquad \forall p\in P,$$

$$0\leq q_{ijv}\leq K_vx_{ijv} \qquad \forall (i,j)\in A,\ v\in V,$$

$$y_p\leq M_pz_p \qquad \forall p\in P.$$

In these constraints, $a_s$ is supplier $s$'s available supply, $r_c$ is customer $c$'s demand, $K_v$ is mode $v$'s capacity, and $M_p$ is a valid upper bound on plant throughput. Here the total customer demand provides that bound. The first constraint allows unused supply; the second requires all demand to be fulfilled; the third balances each plant's incoming and outgoing shipments. The capacity constraint prevents shipment on an unselected link-mode movement and limits the quantity on a selected movement. The final constraint prevents an inactive plant from handling flow.

The code represents $x_{ijv}$ with `routing`, $q_{ijv}$ with `shipment_quantity`, $y_p$ with `facility_handling`, and $z_p$ with `facility_active`. Distance, emissions, and cost factors are stored in the distance matrix and vehicle and node records. The model is a mixed-integer linear program because its objective and constraints are linear and it includes both binary and continuous variables.

## 2. Imports and plotting setup

The model uses NumPy for distance calculations, pandas for displaying the distance matrix, Matplotlib and Seaborn for plots, and PuLP to express and solve the mixed-integer linear program. The code follows the mathematical sequence: define the data, calculate distances, formulate the model, and then inspect the solution.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pulp import (
    LpMinimize,
    LpProblem,
    LpStatus,
    LpVariable,
    PULP_CBC_CMD,
    lpSum,
    value,
)

plt.style.use("default")
sns.set_palette("husl")

## 3. Network and vehicle data

A node stores its location and its supply-chain role. Suppliers have available supply, customers have demand, and plants have fixed daily emissions. A vehicle record stores the capacity and per-kilometre emissions and cost factors for one transportation mode. The concrete example contains three suppliers, two plants, four customers, and two modes.

In [ ]:
@dataclass
class Node:
    """A supplier, plant, or customer in the supply-chain network."""

    id: str
    x: float
    y: float
    node_type: str
    demand: float = 0.0
    supply: float = 0.0
    facility_emissions: float = 0.0


@dataclass
class Vehicle:
    """A transportation mode with capacity, emissions, and cost factors."""

    id: str
    vehicle_type: str
    capacity: float
    emission_factor: float
    cost_per_km: float


@dataclass
class CarbonFootprintProblem:
    """Input data and objective weights for the carbon-footprint model."""

    nodes: List[Node]
    vehicles: List[Vehicle]
    carbon_budget: float
    alpha: float = 0.6
    beta: float = 0.4

## 4. Create the concrete example

The coordinates are the locations used to calculate all link distances. Supplier values are supply limits, customer values are required demand, and plant emissions are charged only when a plant is active. The available supply exceeds total demand, so the model may leave some supply unused. No predetermined optimal footprint is assumed: the result depends on the distances calculated from these coordinates, capacities, weights, and the carbon budget.

In [ ]:
def create_concrete_example() -> CarbonFootprintProblem:
    """Build the example network and its transportation-mode data."""
    nodes = [
        Node("S1", 0, 100, "supplier", supply=100),
        Node("S2", 200, 100, "supplier", supply=120),
        Node("S3", 100, 200, "supplier", supply=80),
        Node("P1", 50, 50, "plant", facility_emissions=50),
        Node("P2", 150, 50, "plant", facility_emissions=75),
        Node("C1", 30, 0, "customer", demand=40),
        Node("C2", 70, 0, "customer", demand=35),
        Node("C3", 130, 0, "customer", demand=45),
        Node("C4", 170, 0, "customer", demand=30),
    ]

    vehicles = [
        Vehicle("Truck", "Truck", 100, 0.8, 1.2),
        Vehicle("Rail", "Rail", 500, 0.3, 0.8),
    ]

    carbon_budget = 1000
    return CarbonFootprintProblem(nodes, vehicles, carbon_budget)


problem = create_concrete_example()
print(
    f"Problem created with {len(problem.nodes)} nodes "
    f"and {len(problem.vehicles)} vehicle modes."
)
print(f"Carbon budget: {problem.carbon_budget} kg CO2e/day")

## 5. Calculate the distance matrix

Distance determines the emissions and cost of each candidate movement. For nodes with coordinates $(x_i,y_i)$ and $(x_j,y_j)$, the Euclidean distance is $d_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}$. The matrix is symmetric, has zero diagonal, and is measured in kilometres under the stated coordinate assumption. Only supplier-to-plant and plant-to-customer entries will later be used as candidate links.

In [ ]:
def calculate_distance(node1: Node, node2: Node) -> float:
    """Return the Euclidean distance between two nodes."""
    return float(np.hypot(node1.x - node2.x, node1.y - node2.y))


def build_distance_matrix(nodes: List[Node]) -> np.ndarray:
    """Build the pairwise Euclidean distance matrix for the network."""
    n = len(nodes)
    distances = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            if i != j:
                distances[i, j] = calculate_distance(nodes[i], nodes[j])

    return distances


distance_matrix = build_distance_matrix(problem.nodes)
node_indices = {node.id: i for i, node in enumerate(problem.nodes)}

print("Distance matrix (first 5 x 5, km):")
print(
    pd.DataFrame(
        distance_matrix[:5, :5],
        index=[node.id for node in problem.nodes[:5]],
        columns=[node.id for node in problem.nodes[:5]],
    ).round(2)
)

## 6. Formulate and solve the mixed-integer model

The model creates candidate links only for supplier-to-plant and plant-to-customer movements. A binary variable selects a link and mode; a continuous variable assigns the shipment quantity to that movement. Linking quantity to the binary decision enforces vehicle capacity and ensures that a movement is charged only when selected. Supplier supply limits, customer demand, plant flow balance, facility activation, and the carbon budget complete the formulation.

The solver receives `distance_matrix` explicitly, so the function uses the distance data passed to it rather than relying on a hidden global value. If CBC does not return an optimal solution, the returned status is retained and solution quantities are not presented as an optimum.

In [ ]:
def solve_carbon_footprint_mip(
    problem: CarbonFootprintProblem,
    distance_matrix: np.ndarray,
) -> Dict:
    """Solve the carbon-footprint supply-chain model with CBC."""
    if distance_matrix.shape != (len(problem.nodes), len(problem.nodes)):
        raise ValueError("The distance matrix must have one row and column per node.")
    if problem.alpha < 0 or problem.beta < 0:
        raise ValueError("Objective weights must be nonnegative.")

    suppliers = [
        i for i, node in enumerate(problem.nodes)
        if node.node_type == "supplier"
    ]
    plants = [
        i for i, node in enumerate(problem.nodes)
        if node.node_type == "plant"
    ]
    customers = [
        i for i, node in enumerate(problem.nodes)
        if node.node_type == "customer"
    ]
    vehicles = range(len(problem.vehicles))
    arcs = (
        [(i, j) for i in suppliers for j in plants]
        + [(i, j) for i in plants for j in customers]
    )

    model = LpProblem("Carbon_Footprint_Optimization", LpMinimize)

    # Each binary variable represents one available link-mode movement.
    routing = {
        (i, j, v): LpVariable(
            f"route_{i}_{j}_{v}", cat="Binary"
        )
        for i, j in arcs
        for v in vehicles
    }
    shipment_quantity = {
        (i, j, v): LpVariable(
            f"quantity_{i}_{j}_{v}", lowBound=0, cat="Continuous"
        )
        for i, j in arcs
        for v in vehicles
    }
    facility_handling = {
        i: LpVariable(f"handling_{i}", lowBound=0, cat="Continuous")
        for i in plants
    }
    facility_active = {
        i: LpVariable(f"active_{i}", cat="Binary")
        for i in plants
    }

    transport_emissions = lpSum(
        routing[i, j, v]
        * distance_matrix[i, j]
        * problem.vehicles[v].emission_factor
        for i, j in arcs
        for v in vehicles
    )
    transport_cost = lpSum(
        routing[i, j, v]
        * distance_matrix[i, j]
        * problem.vehicles[v].cost_per_km
        for i, j in arcs
        for v in vehicles
    )
    facility_emissions = lpSum(
        facility_active[i] * problem.nodes[i].facility_emissions
        for i in plants
    )
    total_emissions = transport_emissions + facility_emissions

    model += (
        problem.alpha * total_emissions
        + problem.beta * transport_cost
    )

    # Available supply is an upper bound; customer demand must be met exactly.
    for i in suppliers:
        outbound_quantity = lpSum(
            shipment_quantity[i, j, v]
            for j in plants
            for v in vehicles
        )
        model += outbound_quantity <= problem.nodes[i].supply

    for j in customers:
        inbound_quantity = lpSum(
            shipment_quantity[i, j, v]
            for i in plants
            for v in vehicles
        )
        model += inbound_quantity == problem.nodes[j].demand

    total_demand = sum(problem.nodes[i].demand for i in customers)
    for i in plants:
        inbound_quantity = lpSum(
            shipment_quantity[j, i, v]
            for j in suppliers
            for v in vehicles
        )
        outbound_quantity = lpSum(
            shipment_quantity[i, j, v]
            for j in customers
            for v in vehicles
        )
        model += inbound_quantity == outbound_quantity
        model += facility_handling[i] == inbound_quantity
        model += facility_handling[i] <= total_demand * facility_active[i]

    # A movement can carry positive flow only when selected, up to its capacity.
    for i, j in arcs:
        for v in vehicles:
            model += (
                shipment_quantity[i, j, v]
                <= problem.vehicles[v].capacity * routing[i, j, v]
            )

    model += total_emissions <= problem.carbon_budget

    print("Solving the mixed-integer model...")
    model.solve(PULP_CBC_CMD(msg=False))
    status = LpStatus[model.status]

    solution = {
        "status": status,
        "objective": None,
        "transport_emissions": None,
        "facility_emissions": None,
        "total_emissions": None,
        "transport_cost": None,
        "routes": [],
        "active_facilities": [],
    }

    if status != "Optimal":
        return solution

    solution.update(
        {
            "objective": value(model.objective),
            "transport_emissions": value(transport_emissions),
            "facility_emissions": value(facility_emissions),
            "total_emissions": value(total_emissions),
            "transport_cost": value(transport_cost),
        }
    )

    for (i, j, v), variable in routing.items():
        quantity = value(shipment_quantity[i, j, v])
        if value(variable) > 0.5 and quantity > 1e-7:
            distance = float(distance_matrix[i, j])
            solution["routes"].append(
                {
                    "from": problem.nodes[i].id,
                    "to": problem.nodes[j].id,
                    "vehicle": problem.vehicles[v].vehicle_type,
                    "distance": distance,
                    "quantity": quantity,
                    "emissions": (
                        distance * problem.vehicles[v].emission_factor
                    ),
                    "cost": distance * problem.vehicles[v].cost_per_km,
                }
            )

    for i, variable in facility_active.items():
        if value(variable) > 0.5:
            solution["active_facilities"].append(problem.nodes[i].id)

    route_emissions = sum(route["emissions"] for route in solution["routes"])
    route_cost = sum(route["cost"] for route in solution["routes"])
    assert np.isclose(route_emissions, solution["transport_emissions"])
    assert np.isclose(route_cost, solution["transport_cost"])
    assert np.isclose(
        solution["transport_emissions"] + solution["facility_emissions"],
        solution["total_emissions"],
    )

    return solution


solution = solve_carbon_footprint_mip(problem, distance_matrix)
print(f"Solution status: {solution['status']}")
if solution["status"] == "Optimal":
    print(f"Weighted objective value: {solution['objective']:.2f}")
else:
    print("No optimal solution is available for reporting.")

## 7. Inspect the solution

The objective value is a weighted combination of emissions and cost, so it is not itself a carbon total. The emissions breakdown reports transportation and facility emissions separately, while the route list shows each selected movement's mode, distance, and shipment quantity. An empty active-facility list means no plant is active in the reported solution; the model's demand and flow constraints still require all customer demand to pass through plants.

In [ ]:
def analyze_solution(
    problem: CarbonFootprintProblem,
    solution: Dict,
) -> None:
    """Display the objective, emissions, costs, and selected movements."""
    print("=" * 60)
    print("CARBON FOOTPRINT OPTIMIZATION RESULTS")
    print("=" * 60)
    print(f"\nSolution status: {solution['status']}")

    if solution["status"] != "Optimal":
        print("No optimal solution is available for analysis.")
        return

    print(f"Weighted objective value: {solution['objective']:.2f}")
    print("\n--- EMISSIONS AND COST ---")
    print(
        "Transportation emissions: "
        f"{solution['transport_emissions']:.2f} kg CO2e/day"
    )
    print(
        "Facility emissions: "
        f"{solution['facility_emissions']:.2f} kg CO2e/day"
    )
    print(
        f"Total emissions: {solution['total_emissions']:.2f} kg CO2e/day"
    )
    print(f"Transportation cost: ${solution['transport_cost']:.2f}")
    print(f"Carbon budget: {problem.carbon_budget:.2f} kg CO2e/day")

    print("\n--- SELECTED SHIPMENT MOVEMENTS ---")
    if solution["routes"]:
        for route in solution["routes"]:
            print(
                f"{route['from']} -> {route['to']} ({route['vehicle']}): "
                f"{route['quantity']:.2f} units"
            )
            print(
                f"  Distance: {route['distance']:.2f} km; "
                f"emissions: {route['emissions']:.2f} kg CO2e; "
                f"cost: ${route['cost']:.2f}"
            )
    else:
        print("No shipment movements are selected.")

    print("\n--- ACTIVE PLANTS ---")
    if solution["active_facilities"]:
        nodes_by_id = {node.id: node for node in problem.nodes}
        for facility_id in solution["active_facilities"]:
            emissions = nodes_by_id[facility_id].facility_emissions
            print(f"{facility_id}: {emissions:.2f} kg CO2e/day")
    else:
        print("No plants are active.")


analyze_solution(problem, solution)

## 8. Visualize the network and emissions

The network panel shows node roles and the direction, mode, and quantity of selected movements. The other panels summarize emissions, movement distances by mode, and emissions per kilometre for each selected movement. Because emissions and cost factors are fixed per kilometre in this model, a movement's emissions per kilometre correspond to its selected mode's factor. The figures describe the model solution; they do not add constraints or alter the optimization.

In [ ]:
def visualize_solution(
    problem: CarbonFootprintProblem,
    solution: Dict,
) -> None:
    """Plot the network solution and its emissions summaries."""
    if solution["status"] != "Optimal":
        print("Visualization requires an optimal solution.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("Carbon Footprint Optimization Analysis", fontsize=16)

    # Node markers distinguish suppliers, plants, and customers.
    ax1 = axes[0, 0]
    node_styles = {
        "supplier": ("green", "^", 150),
        "plant": ("orange", "s", 200),
        "customer": ("blue", "o", 100),
    }
    for node in problem.nodes:
        color, marker, size = node_styles[node.node_type]
        ax1.scatter(
            node.x,
            node.y,
            c=color,
            marker=marker,
            s=size,
            alpha=0.75,
            edgecolors="black",
            linewidth=1,
            label=node.node_type.title(),
        )
        ax1.annotate(
            node.id,
            (node.x, node.y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )

    node_by_id = {node.id: node for node in problem.nodes}
    vehicle_colors = {"Truck": "red", "Rail": "purple"}
    for route in solution["routes"]:
        start = node_by_id[route["from"]]
        end = node_by_id[route["to"]]
        color = vehicle_colors.get(route["vehicle"], "black")
        ax1.annotate(
            "",
            xy=(end.x, end.y),
            xytext=(start.x, start.y),
            arrowprops={
                "arrowstyle": "->",
                "color": color,
                "lw": 1.5,
                "alpha": 0.8,
            },
        )

    for node_type in node_styles:
        color, marker, size = node_styles[node_type]
        ax1.scatter([], [], c=color, marker=marker, s=size, label=node_type.title())
    for vehicle_type, color in vehicle_colors.items():
        ax1.plot([], [], color=color, label=f"{vehicle_type} movement")
    ax1.set_xlabel("X coordinate (km)")
    ax1.set_ylabel("Y coordinate (km)")
    ax1.set_title("Selected Supply-Chain Movements")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="best", fontsize=8)

    # The emissions panel separates variable transport emissions from fixed plant emissions.
    ax2 = axes[0, 1]
    emissions_data = [
        ("Transportation", solution["transport_emissions"]),
        ("Facilities", solution["facility_emissions"]),
    ]
    bars = ax2.bar(
        [item[0] for item in emissions_data],
        [item[1] for item in emissions_data],
        color=["lightcoral", "lightsalmon"],
        alpha=0.85,
        edgecolor="black",
    )
    ax2.set_ylabel("Emissions (kg CO2e/day)")
    ax2.set_title("Emissions Breakdown")
    ax2.grid(True, alpha=0.3, axis="y")
    for bar, (_, amount) in zip(bars, emissions_data):
        ax2.annotate(
            f"{amount:.1f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    # A secondary axis keeps distance and emissions in their own units.
    ax3 = axes[1, 0]
    vehicle_emissions = {}
    vehicle_distances = {}
    for route in solution["routes"]:
        vehicle = route["vehicle"]
        vehicle_emissions[vehicle] = (
            vehicle_emissions.get(vehicle, 0.0) + route["emissions"]
        )
        vehicle_distances[vehicle] = (
            vehicle_distances.get(vehicle, 0.0) + route["distance"]
        )

    vehicle_types = list(vehicle_emissions)
    positions = np.arange(len(vehicle_types))
    ax3_twin = ax3.twinx()
    ax3.bar(
        positions - 0.2,
        [vehicle_emissions[vehicle] for vehicle in vehicle_types],
        0.4,
        label="Emissions (kg CO2e)",
        color="red",
        alpha=0.7,
    )
    ax3_twin.bar(
        positions + 0.2,
        [vehicle_distances[vehicle] for vehicle in vehicle_types],
        0.4,
        label="Distance (km)",
        color="blue",
        alpha=0.7,
    )
    ax3.set_xlabel("Vehicle mode")
    ax3.set_ylabel("Emissions (kg CO2e)", color="red")
    ax3_twin.set_ylabel("Distance (km)", color="blue")
    ax3.set_title("Emissions and Distance by Mode")
    ax3.set_xticks(positions)
    ax3.set_xticklabels(vehicle_types)
    ax3.tick_params(axis="y", labelcolor="red")
    ax3_twin.tick_params(axis="y", labelcolor="blue")
    handles1, labels1 = ax3.get_legend_handles_labels()
    handles2, labels2 = ax3_twin.get_legend_handles_labels()
    if handles1 or handles2:
        ax3.legend(handles1 + handles2, labels1 + labels2, loc="best")

    ax4 = axes[1, 1]
    route_labels = [
        f"{route['from']} to {route['to']}\n{route['vehicle']}"
        for route in solution["routes"]
    ]
    route_efficiency = [
        route["emissions"] / route["distance"]
        for route in solution["routes"]
        if route["distance"] > 0
    ]
    if route_labels:
        ax4.bar(
            route_labels,
            route_efficiency,
            color=[
                "lightcoral" if route["vehicle"] == "Truck" else "lightblue"
                for route in solution["routes"]
                if route["distance"] > 0
            ],
            alpha=0.85,
            edgecolor="black",
        )
    ax4.set_ylabel("Emissions per kilometre (kg CO2e/km)")
    ax4.set_title("Movement Emissions Factors")
    ax4.grid(True, alpha=0.3, axis="y")
    plt.setp(ax4.get_xticklabels(), rotation=35, ha="right")

    plt.tight_layout()
    plt.show()


visualize_solution(problem, solution)

## 9. Carbon-budget sensitivity analysis

A sensitivity analysis changes one input at a time while keeping the network, vehicle data, and objective weights fixed. Here, each trial changes only the carbon budget $B$ and solves the same mixed-integer model. A tighter budget may make the problem infeasible or may force a different combination of movements and active plants. A looser budget does not require the solution to use all available emissions: the objective still values emissions reduction.

The plots and table are generated from solver results when the notebook runs. Only trials with an optimal status are plotted as feasible optima; other solver statuses are reported without treating missing values as results.

In [ ]:
def sensitivity_analysis(
    problem: CarbonFootprintProblem,
    distance_matrix: np.ndarray,
) -> List[Dict]:
    """Solve the same model over a range of carbon-budget values."""
    print("=" * 60)
    print("CARBON-BUDGET SENSITIVITY ANALYSIS")
    print("=" * 60)

    budgets = [200, 400, 600, 800, 1000, 1200]
    results = []
    for budget in budgets:
        test_problem = CarbonFootprintProblem(
            nodes=problem.nodes,
            vehicles=problem.vehicles,
            carbon_budget=budget,
            alpha=problem.alpha,
            beta=problem.beta,
        )
        result = solve_carbon_footprint_mip(test_problem, distance_matrix)
        results.append(
            {
                "budget": budget,
                "total_emissions": result["total_emissions"],
                "transport_cost": result["transport_cost"],
                "objective": result["objective"],
                "status": result["status"],
            }
        )

    feasible_results = [
        result for result in results if result["status"] == "Optimal"
    ]
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Sensitivity to the Carbon Budget", fontsize=16)

    if feasible_results:
        budgets_feasible = [result["budget"] for result in feasible_results]
        emissions_feasible = [
            result["total_emissions"] for result in feasible_results
        ]
        costs_feasible = [
            result["transport_cost"] for result in feasible_results
        ]
        objectives_feasible = [
            result["objective"] for result in feasible_results
        ]

        axes[0, 0].plot(
            budgets_feasible,
            emissions_feasible,
            "bo-",
            linewidth=2,
            markersize=7,
            label="Optimal emissions",
        )
        axes[0, 0].plot(
            budgets_feasible,
            budgets_feasible,
            "r--",
            alpha=0.7,
            label="Budget limit",
        )
        axes[0, 0].set_xlabel("Carbon budget (kg CO2e/day)")
        axes[0, 0].set_ylabel("Emissions (kg CO2e/day)")
        axes[0, 0].set_title("Emissions and Budget")
        axes[0, 0].grid(True, alpha=0.3)
        axes[0, 0].legend()

        axes[0, 1].plot(
            budgets_feasible,
            costs_feasible,
            "go-",
            linewidth=2,
            markersize=7,
        )
        axes[0, 1].set_xlabel("Carbon budget (kg CO2e/day)")
        axes[0, 1].set_ylabel("Transportation cost")
        axes[0, 1].set_title("Transportation Cost by Budget")
        axes[0, 1].grid(True, alpha=0.3)

        axes[1, 0].plot(
            budgets_feasible,
            objectives_feasible,
            "mo-",
            linewidth=2,
            markersize=7,
        )
        axes[1, 0].set_xlabel("Carbon budget (kg CO2e/day)")
        axes[1, 0].set_ylabel("Weighted objective value")
        axes[1, 0].set_title("Weighted Objective by Budget")
        axes[1, 0].grid(True, alpha=0.3)

        points = axes[1, 1].scatter(
            emissions_feasible,
            costs_feasible,
            c=budgets_feasible,
            cmap="viridis",
            s=90,
        )
        axes[1, 1].set_xlabel("Emissions (kg CO2e/day)")
        axes[1, 1].set_ylabel("Transportation cost")
        axes[1, 1].set_title("Cost-Emission Outcomes")
        axes[1, 1].grid(True, alpha=0.3)
        colorbar = fig.colorbar(points, ax=axes[1, 1])
        colorbar.set_label("Carbon budget (kg CO2e/day)")
    else:
        for axis in axes.flat:
            axis.text(
                0.5,
                0.5,
                "No budget trial returned an optimal solution.",
                ha="center",
                va="center",
                transform=axis.transAxes,
            )
            axis.set_axis_off()

    plt.tight_layout()
    plt.show()

    print("\nSensitivity analysis results:")
    print(f"{'Budget':<12} {'Emissions':<16} {'Cost':<14} {'Status':<12}")
    print("-" * 58)
    for result in results:
        if result["status"] == "Optimal":
            print(
                f"{result['budget']:<12} "
                f"{result['total_emissions']:<16.2f} "
                f"{result['transport_cost']:<14.2f} "
                f"{result['status']:<12}"
            )
        else:
            print(
                f"{result['budget']:<12} {'N/A':<16} "
                f"{'N/A':<14} {result['status']:<12}"
            )

    return results


sensitivity_results = sensitivity_analysis(problem, distance_matrix)

## 10. Interpretation and limitations

The formulation makes the emissions-cost trade-off explicit: emissions and transportation cost contribute to the weighted objective, while the carbon budget is a separate feasibility requirement. The shipment quantities explain how the selected movements meet customer demand through plants; the active-plant list identifies which fixed facility emissions are included. Sensitivity results show how the optimal solution changes as the permitted emissions level changes, subject to the same network and data assumptions.

The formulation is exact for the stated mixed-integer model when the solver reports an optimal status. Its conclusions remain conditional on the model scope. In particular, one selected link-mode movement represents at most one vehicle movement, vehicle capacity is applied to that movement, and the per-kilometre emissions factor does not vary with shipment quantity. The model does not represent vehicle tours, repeated trips, production, or time-dependent operations. These assumptions should be considered when interpreting results or extending the network.